# Capstone — Which visible pages deserve the editor's first pass?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROHIT-25607/FLYRANK-INTERN/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This notebook **is** the research paper, in executable form. It mirrors the deployed page
(`docs/index.html`, GitHub Pages) section by section and regenerates every chart the paper embeds:

- Section 1 Question (the decision this supports)
- Section 2 Data (release, windows, exclusions — public-safe)
- Section 3 Methodology (label, features, baseline, validation, leakage checks)
- Section 4 Results (model vs baseline on the same split)
- Section 5 Limitations (what this work cannot claim)
- Section 6 Ranked recommendations (the action playbook)
- Section 7 Artifacts the paper embeds (charts + receipts, refreshed into `docs/img/`)

It is one clean pipeline built from the weekly trail: w03 data contract, w04 baseline rule,
w05 model + grouped split, w06 leakage audit, w07 playbook. Numbers match the committed receipts
(`work/outputs/w0X_*_metrics.json`). Care words throughout: *observed, measured, directional,
decision-support* — nothing here "proves" or "causes" anything about Google's algorithm.

In [1]:
import os, json, duckdb, pandas as pd, numpy as np
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

root = Path(".").resolve()
while root != root.parent and not (root / "AGENTS.md").exists():
    root = root.parent
OUT = root / "work" / "outputs"
CACHE = root / "work" / "outputs" / "w03_cache"
DOCIMG = root / "docs" / "img"
DOCIMG.mkdir(parents=True, exist_ok=True)

import sklearn, sys
print("python", sys.version.split()[0], "| duckdb", duckdb.__version__,
      "| pandas", pd.__version__, "| sklearn", sklearn.__version__)

def hf_token():
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        try:
            from google.colab import userdata
            tok = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if not tok:
        import getpass
        tok = getpass.getpass("HF_TOKEN: ")
    return tok

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '" + hf_token() + "')")
con.execute("SET http_timeout = 900")
REL = "hf://datasets/FlyRank/internship-warehouse"
DEC, LAB = "2026-03", "2026-04"

def month_path(month):  # local cache when present (fast), else the warehouse partition
    name = {"2026-03": "fact_2026-03.parquet", "2026-04": "fact_2026-04.parquet"}[month]
    p = CACHE / name
    if p.exists():
        return f"read_parquet('{p.as_posix()}')"
    return f"read_parquet('{REL}/fact_content_daily_performance/month={month}/data_0.parquet')"

ff = con.sql(f'''
    WITH dec AS (
      SELECT * FROM {month_path(DEC)} WHERE gsc_data_available IS TRUE
    )
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_march,
           SUM(gsc_clicks)      AS clk_march,
           AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0) AS pos_march,
           COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS active_days_march,
           SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-25') AS imp_last7,
           MAX(report_date) FILTER (WHERE gsc_impressions > 0) AS last_active_day
    FROM dec GROUP BY 1, 2
''').df()

lab = con.sql(f'''
    SELECT content_hash_id, SUM(gsc_impressions) AS apr_imp
    FROM {month_path(LAB)} WHERE gsc_data_available IS TRUE GROUP BY 1
''').df()

DEC_END = pd.Timestamp("2026-03-31")
df = ff.merge(lab, on="content_hash_id", how="inner")
df = df[df["imp_march"] >= 100].reset_index(drop=True)
df = df.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
df["declined_next_30d"] = (df["apr_imp"] < 0.8 * df["imp_march"]).astype(int)
df["ctr"] = df["clk_march"] / df["imp_march"]
df["momentum_last7"] = (df["imp_last7"] / df["imp_march"]).fillna(0.0)
df["inert_days"] = (DEC_END - pd.to_datetime(df["last_active_day"])).dt.days.fillna(30).astype(float)

def pos_band(p):
    if pd.isna(p) or p <= 0: return "unpositioned"
    if p <= 3:  return "top3"
    if p <= 10: return "p1"
    if p <= 20: return "p2"
    return "deep"
df["band"] = df["pos_march"].map(pos_band)
df["pos_valid"] = df["pos_march"].notna().astype(int)

print("lane pool :", len(df), "pages |", df["client_hash_id"].nunique(), "clients | base",
      round(df["declined_next_30d"].mean(), 3))

FEATS = ["imp_march", "clk_march", "ctr", "pos_march", "active_days_march",
         "momentum_last7", "inert_days", "band", "pos_valid"]


python 3.12.0 | duckdb 1.5.5 | pandas 2.2.3 | sklearn 1.6.1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

lane pool : 100893 pages | 43 clients | base 0.515


## 1. Question

**The decision this supports.** A FlyRank editor opens the month with a budget of a few hundred page
reviews across a whole client portfolio. *Which pages, in which order, should they open first?*

**The naive answer fails (the tension).** "Review the biggest traffic fallers" scores barely above
chance on held-out clients (AUC 0.451) — a page that just crashed is often a page that was already
invisible. Ranking quality beats recency of the loss.

**The claim ladder, stated before any result:**
- **Observed** (what happened in the data): visible pages with low recent momentum and weak CTR-for-band
  declined next month more often, on this 100k-page March slice.
- **Measured** (the only %, the only holdout): held-out precision@K of the forest's ranking, on 13
  clients never seen in training.
- **Directional / decision-support** (everything else): the queue says *review these first*, never
  *refresh and you'll recover*. We did not run a content experiment and make no causal claim.
- **Not claimed, ever:** no proof about Google's algorithm, no forecast of future traffic values.

In [2]:
# Claim ladder, in code: make sure the notebook's variables mean what the paper says.
measured_claim = "held-out P@K + AUC on 13 clients not seen in training (below)"
decision_support = "full-pool queue: review these first, in this order (Section 6)"
causal_claim = "NOT measured — no experiment was run (Section 5)"
print("measured      :", measured_claim)
print("decision-sup  :", decision_support)
print("causal claim  :", causal_claim)


measured      : held-out P@K + AUC on 13 clients not seen in training (below)
decision-sup  : full-pool queue: review these first, in this order (Section 6)
causal claim  : NOT measured — no experiment was run (Section 5)


## 2. Data

- **Release:** `FlyRank/internship-warehouse` on Hugging Face (public-but-gated, read token).
  Table: `fact_content_daily_performance`, partitioned by `month=YYYY-MM` — per-page daily counts of
  impressions, clicks, average position, active days, content-release date. Anonymized: every
  identity in this notebook is a hash; no client names, domains, URLs, or raw queries.
- **Windows:** features from the closed **2026-03** partition; the outcome label is measured on the
  **2026-04** partition. Adjacent, non-overlapping — no window ever contains both a decision and its
  outcome.
- **Lane pool:** 100,893 pages across 43 clients, March impressions ≥ 100, adequate Search Console
  coverage (`gsc_data_available`).
- **Excluded and why:** GA4 engagement facts (sparse — ~4% coverage; they would silently shrink the
  real population); country/device/type dimension slices (too granular to aggregate safely);
  content-release date as a feature (as-of-release, leaks the future);
  the sealed June 2026 `_sample` month (held for the next cycle, never used for label development).
- The paper quotes this same slice; the numbers below must match `w05_model_metrics.json`.

In [3]:
# The pool just built in the setup cell IS Section 2's data. Verify it against the receipts.
w05 = json.load(open(OUT / "w05_model_metrics.json", encoding="utf-8"))
print("lane pool:", len(df), "pages | decision window:", DEC, "| label window:", LAB)
print("receipt cross-check:", "test_rows =", w05["test_rows"],
      "| test_base_rate =", w05["test_base_rate"])
print("  nothing from", "2026-06 (_sample) is touched anywhere in this notebook")


lane pool: 100893 pages | decision window: 2026-03 | label window: 2026-04
receipt cross-check: test_rows = 20002 | test_base_rate = 0.429
  nothing from 2026-06 (_sample) is touched anywhere in this notebook


## 3. Methodology

**Assumptions.** Positions/CTR are monthly aggregates (a 31-day average hides intra-month slides — the
limitations section owns this). Monthly impression demand is a reasonable proxy for "visibility". A page
that stays within 80% of its own March level is "stable"; below that is "declined" — a *relative* label,
robust to market-wide shifts and the label editors actually act on.

**Label.** `declined_next_30d` = April impressions < 0.8 × March impressions (April only, never a feature).

**Features (9, all March-only).** impressions, clicks, CTR, average position, active days, last-7-day
impression share (`momentum_last7` — the decay sensor), days since last active day (`inert_days`),
position band (ordinal), position-present flag. Every feature ends 2026-03-31.

**Baseline.** The w04 hand rule (position-band × CTR-tercile risk × log1p impressions) with per-band CTR
tercile edges fit on March *features* (no labels), as shipped in w05. A robustness probe re-fits those
edges on the 30 training clients only — the P@50 result is unchanged, so the rule's calibration generalizes.

**Models.** Logistic regression (scaled) and Random Forest (400 trees, `min_samples_leaf=15`,
`max_features=sqrt`, seed 42), class-weighted for the ~49/51 split.

**Validation design.**
- **Grouped split:** 30 clients train / 13 held-out (`GroupShuffleSplit`, seed 42, 30% test). Pages are
  never split randomly — that would leak client behavior; w06 showed a random split reaches P@50 0.92
  only by reporting 39/39 previously-seen clients in its top-50.
- **Leakage checks (w03 + w06):** feature/label window separation; feature-staleness; redundant-column
  red flags (content-release date, event timestamps); a deliberate *leak trap* (inject an April-view
  feature → P@50 1.00, AUC 0.995 — the exact red flag honest systems must never show); overlap audit
  (random-split top-50 overlap 39/39 vs grouped 0/13).
- **Stability:** 5-fold GroupKFold → P@50 0.944 ± 0.023, AUC 0.751 ± 0.024.

In [4]:
X = df[FEATS + ["declined_next_30d", "client_hash_id"]].copy()
Xnum = X.select_dtypes(include=[np.number]).drop(columns=["declined_next_30d"])
Xnum = Xnum.fillna({"pos_march": 0.0, "momentum_last7": 0.0})
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
band_enc = OrdinalEncoder(dtype=float).fit_transform(X[["band"]].to_numpy().reshape(-1, 1)).ravel()
X_model = Xnum.assign(band=band_enc.astype(float)).astype(float)
y = X["declined_next_30d"].astype(int).reset_index(drop=True)
groups = X["client_hash_id"].reset_index(drop=True)

from sklearn.model_selection import GroupShuffleSplit
tr_i, te_i = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
                  .split(X_model, y, groups))
print("train:", len(tr_i), "pages /", groups.iloc[tr_i].nunique(), "clients | test:",
      len(te_i), "pages /", groups.iloc[te_i].nunique(), "clients | test base",
      round(float(y.iloc[te_i].mean()), 3))
sc = StandardScaler().fit(X_model.iloc[tr_i])
X_tr_s, X_te_s = sc.transform(X_model.iloc[tr_i]), sc.transform(X_model.iloc[te_i])
print("scaler fit on train only; split fixed (seed 42) -> table reproduces on rerun")

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

lr = LogisticRegression(max_iter=3000, random_state=42).fit(X_tr_s, y.iloc[tr_i])
rf = RandomForestClassifier(n_estimators=400, min_samples_leaf=15, max_features="sqrt",
                            random_state=42, n_jobs=-1).fit(X_model.iloc[tr_i], y.iloc[tr_i])

def baseline_from(pool):  # w04 rule calibration: per-band CTR tercile edges, features only
    edges = {}
    for b in ["top3", "p1", "p2", "deep"]:
        m = pool["band"].eq(b) & (pool["imp_march"] >= 500)
        if m.sum() < 3: continue
        q = np.quantile(pool.loc[m, "ctr"].to_numpy(), [1 / 3, 2 / 3])
        edges[b] = (float(q[0]), float(q[1]))
    return edges

def baseline_score(pages, edges):
    risk = np.zeros(len(pages))
    ctr = pages["ctr"]
    for b, (lo, hi) in edges.items():
        sel = pages["band"].eq(b).to_numpy() & pages["imp_march"].ge(500).to_numpy()
        if b in ("top3", "p1"):
            risk[sel & (ctr < lo).to_numpy()] = 3.0 if b == "top3" else 2.5
            risk[sel & ctr.ge(lo).to_numpy() & ctr.lt(hi).to_numpy()] = 2.0 if b == "top3" else 1.5
        risk[sel & (ctr.ge(lo).to_numpy() if b in ("p2", "deep") else ctr.ge(hi).to_numpy())] = 1.0
    return risk * np.log1p(pages["imp_march"].to_numpy())

edges_tr = baseline_from(df.iloc[tr_i])          # robustness probe: TRAINING-clients-only calibration
edges = baseline_from(df)                        # as shipped in w05: whole-pool feature calibration
scores = {
    "baseline rule": -baseline_score(df.iloc[te_i], edges),
    "logistic regression": -lr.predict_proba(X_te_s)[:, 1],
    "random forest": -rf.predict_proba(X_model.iloc[te_i])[:, 1],
}

y_te = y.iloc[te_i].values
_probe = -baseline_score(df.iloc[te_i], edges_tr)
print("robustness probe: rule re-fit on TRAINING clients only keeps P@50 at",
      round(float(y_te[np.argsort(_probe)[:50]].mean()), 3), "(edge calibration generalizes)")
base_te = round(float(y_te.mean()), 3)
print("held-out test set | base rate:", base_te)
from sklearn.metrics import roc_auc_score
header = f"{'method':<22}" + "".join(f" P@{k:<6}" for k in [10, 20, 50, 100]) + "  ROC-AUC"
print(header); print("-" * len(header))
table = {}
for name, s in scores.items():
    order = np.argsort(s)
    pk = {k: round(float(y_te[order[:k]].mean()), 3) for k in [10, 20, 50, 100]}
    auc = round(float(roc_auc_score(y_te, -s)), 3)
    table[name] = {"precision_at_k": {str(k): pk[k] for k in pk}, "roc_auc": auc}
    print(f"{name:<22}" + "".join(f" {pk[k]:<8.3f}" for k in [10, 20, 50, 100]) + f"  {auc:.3f}")


train: 80891 pages / 30 clients | test: 20002 pages / 13 clients | test base 0.429
scaler fit on train only; split fixed (seed 42) -> table reproduces on rerun


robustness probe: rule re-fit on TRAINING clients only keeps P@50 at 0.44 (edge calibration generalizes)
held-out test set | base rate: 0.429
method                 P@10     P@20     P@50     P@100     ROC-AUC
-------------------------------------------------------------------
baseline rule          0.700    0.700    0.440    0.450     0.451
logistic regression    0.800    0.850    0.700    0.690     0.731
random forest          0.900    0.900    0.860    0.870     0.739


## 4. Results (model vs baseline, same split)

Same 13 held-out clients, base rate visible, seed-42 grouped split:

| model (held-out test) | P@10 | P@20 | P@50 | P@100 | AUC |
|---|---|---|---|---|---|
| chance (base rate) | 0.43 | 0.43 | 0.43 | 0.43 | 0.50 |
| rule baseline (w04 rule, retrained edges) | 0.70 | 0.70 | 0.44 | 0.45 | 0.451 |
| logistic regression | 0.80 | 0.85 | 0.70 | 0.69 | 0.731 |
| **random forest** | **0.90** | **0.90** | **0.86** | **0.87** | **0.739** |

The table is *recomputed from scratch below* — if it prints different numbers, the paper is out of sync
with the code, and that is a bug to fix before deploying.

In [5]:
# Section 4 in one place: fill the table object with the printed numbers, then draw the paper's figure.
w05 = json.load(open(OUT / "w05_model_metrics.json", encoding="utf-8"))
for name in table:
    assert table[name] == w05["table"][name], f"capstone differs from w05 receipt for {name}"
print("capstone table == committed w05 receipt: True  (co-signed, reproducible)")

ks = [10, 20, 50, 100]
colors = {"baseline rule": "#9AA0A6", "logistic regression": "#F4B400", "random forest": "#4285F4"}
fig, ax = plt.subplots(figsize=(7.5, 4.6))
for name in scores:
    pk = table[name]["precision_at_k"]
    auc = table[name]["roc_auc"]
    ax.plot(ks, [pk[str(k)] for k in ks], marker="o", lw=2, color=colors[name],
            label=f"{name} (AUC {auc})")
ax.axhline(y_te.mean(), color="gray", ls="--", lw=1.5)
ax.text(100, y_te.mean() + 0.012, f"held-out base rate {y_te.mean():.3f}", ha="right", color="gray")
ax.set_xlabel("K — pages reviewed before the editor stops"); ax.set_ylabel("precision@K")
ax.set_title("Ranking quality on the same 13 held-out clients (seed-42 grouped split)")
ax.set_xticks(ks); ax.grid(alpha=0.3); ax.legend(fontsize=9)
fig.tight_layout(); fig.savefig(DOCIMG / "results_precision_at_k.png", dpi=160)

fig2, ax2 = plt.subplots(figsize=(7, 4))
rfk = [10, 20, 50, 100, 200, 500]
rf_order = np.argsort(scores["random forest"])
rf_pk = {k: float(y_te[rf_order[:k]].mean()) for k in rfk}
ax2.plot(rfk, [rf_pk[k] for k in rfk], marker="o", color="#4285F4")
ax2.axhline(y_te.mean(), color="gray", ls="--", label=f"held-out base {y_te.mean():.3f}")
ax2.set_xlabel("K (pages reviewed)"); ax2.set_ylabel("precision@K")
ax2.set_title("RF ranking, measured on 13 held-out clients")
ax2.legend(); ax2.grid(alpha=0.3)
fig2.tight_layout(); fig2.savefig(DOCIMG / "w07_heldout_pk.png", dpi=150)
print("wrote docs/img/results_precision_at_k.png and docs/img/w07_heldout_pk.png")

from sklearn.inspection import permutation_importance
imp = permutation_importance(rf, X_model.iloc[te_i], y_te, n_repeats=3, random_state=42, n_jobs=-1)
order = np.argsort(-imp.importances_mean)
print("\nRF permutation importance (held-out clients):")
for i in order[:5]:
    print(f"  {X_model.columns[i]:<20} {imp.importances_mean[i]:+.4f}")
json.dump({"capstone": "co-signed with w05", "test_base_rate": base_te,
           "measured_claims": {"precision_at_k": {k: table["random forest"]["precision_at_k"][str(k)] for k in ks},
                               "roc_auc_rf": table["random forest"]["roc_auc"]},
           "top_feature": str(X_model.columns[int(order[0])])},
          open(OUT / "capstone_results.json", "w"), indent=2)
print("\ncapstone_results.json written — traces every number on the paper's Results page")


capstone table == committed w05 receipt: True  (co-signed, reproducible)


wrote docs/img/results_precision_at_k.png and docs/img/w07_heldout_pk.png



RF permutation importance (held-out clients):
  momentum_last7       +0.1354
  ctr                  +0.0120
  pos_march            +0.0072
  active_days_march    +0.0064
  inert_days           +0.0057

capstone_results.json written — traces every number on the paper's Results page


## 5. Limitations (written before a reader writes them for us)

- **Decision support, not a forecast.** Every number on the page is a ranking signal. Nothing predicts a
  future value for any page; nothing claims Google's algorithm is "worked out".
- **Observed → directional.** The *ranking* is measured (held-out P@K — one percentage, one holdout). The
  "editor reviews, then traffic recovers" half was never measured — it would need a randomized treatment
  experiment we didn't run. The page says so.
- **Scores are not probabilities.** We rank within an April-labeled slice; the queue must not be reused
  after the next month closes (monitoring in w07 covers this).
- **Monthly averages hide intra-month slides.** Position 6 on March 31 beats position 6 on March 1, and
  our 31-day average flattens the story — momentum partially rescues this, not completely.
- **Survivor pool.** Invisible pages (impressions < 100) were out of scope; the playbook reviews pages
  *already being seen* but not earning it.
- **One season, one market.** 43 clients, a single client-holdout; GroupKFold stability helps, cross-month
  and cross-market replication is the honest next step.

In [6]:
# Honest-framing self-audit: every claim the page makes maps to exactly one of these.
audit = {
    "measured": ["held-out P@K {0.90, 0.90, 0.86, 0.87, 0.875, 0.844}", "AUC 0.739 on 13 held-out clients"],
    "observed": ["visible pages with low recent momentum + weak band-CTR declined next month more often"],
    "directional": ["momentum_last7 is the RF's top permutation-importance driver"],
    "decision-support": ["the queue says review these first; no auto-refresh, no automation"],
    "NOT_claimed": ["causal refresh lift", "traffic forecast", "Google algorithm insight"],
}
for k, v in audit.items():
    print(f"{k:<16}", " | ", " ; ".join(v))
print("\nNo-go list from w07 applies to everything below: no auto-rewrite/publish/delete,")
print("no label in features, no silent retrain, sealed test month never touched.")


measured          |  held-out P@K {0.90, 0.90, 0.86, 0.87, 0.875, 0.844} ; AUC 0.739 on 13 held-out clients
observed          |  visible pages with low recent momentum + weak band-CTR declined next month more often
directional       |  momentum_last7 is the RF's top permutation-importance driver
decision-support  |  the queue says review these first; no auto-refresh, no automation
NOT_claimed       |  causal refresh lift ; traffic forecast ; Google algorithm insight

No-go list from w07 applies to everything below: no auto-rewrite/publish/delete,
no label in features, no silent retrain, sealed test month never touched.


## 6. Ranked recommendations — the action playbook

Every page in the pool gets a score; scores become four actions; flags become human-readable reason codes.

| action | share of pool | what the editor does |
|---|---|---|
| **investigate_first** | 1.0% (1,009 pages) | top pages seen but under-earning; review the top hundred in one sitting |
| watch_prepare | 1.5% (1,514) | borderline band: draft a refresh while evidence matures |
| monitor | 7.5% (7,567) | drift-watch; re-check when the next month closes |
| no_action | 90.0% (90,803) | leave it — the queue must stay small to stay useful |

Reason codes are feature-only flags (CTR gap vs band, lost momentum, inactivity, volume exposure).
The most common code on review-first pages is `ctr_gap + momentum_loss`: page-one, under-earning clicks,
already decelerating — the page an editor should open first.

In [7]:
# Reasons + archetypes + the four actions (full pool = decision-support; the only measured
# claim remains Section 4's held-out P@K).
band_cmed = df.groupby("band")["ctr"].transform("median")
flags = pd.DataFrame({
    "ctr_gap": df["band"].isin(["top3", "p1"]) & (df["ctr"] <= band_cmed * 0.5),
    "momentum_loss": df["momentum_last7"] <= 0.15,
    "inertia": df["inert_days"] >= 7,
    "volume_exposure": df["imp_march"] >= 10_000,
})
arche = np.select(
    [flags["ctr_gap"],
     flags["momentum_loss"] & (df["active_days_march"] >= 20),
     df["band"].eq("deep") & (flags["momentum_loss"] | flags["inertia"]),
     flags["volume_exposure"],
     (df["momentum_last7"] >= 0.85) & (df["ctr"] > band_cmed)],
    ["visible_low_ctr", "decaying_momentum", "faded_deep", "high_volume_flat", "stable_strong"],
    default="watch_rest")
df["archetype"] = arche

prio = ["ctr_gap", "momentum_loss", "inertia", "volume_exposure"]
def rc(i):
    hits = [c for c in prio if flags.loc[i, c]][:2]
    return "+".join(hits) if hits else "base_anchor"
df["reason_code"] = [rc(i) for i in range(len(df))]

df["rf_score"] = rf.predict_proba(X_model)[:, 1]   # full pool, decision-support
q = df["rf_score"].rank(pct=True)
df["action"] = np.select([q >= 0.99, q >= 0.975, q >= 0.90],
                         ["investigate_first", "watch_prepare", "monitor"], default="no_action")

print("action mix (full pool):")
print(df["action"].value_counts().to_string())
print("\ntop reason codes across the investigate_first tier:")
print(df.loc[df["action"] == "investigate_first", "reason_code"].value_counts().head(5).to_string())
print("\nreview-first position mix:", {k: round(float(v), 3) for k, v in
      df.loc[df["action"] == "investigate_first", "band"].value_counts(normalize=True).items()})

mix = df["action"].value_counts(normalize=True).mul(100)
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(mix.index, mix.values, color="#4C72B0")
ax.set_ylabel("% of pool"); ax.set_title("Action mix (full pool, decision-support)")
for b, v in zip(bars, mix.values):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}%", ha="center", va="bottom")
ax.grid(alpha=0.3, axis="y")
fig.tight_layout(); fig.savefig(DOCIMG / "w07_action_mix.png", dpi=150)
print("\nwrote docs/img/w07_action_mix.png")

queue_cols = set(FEATS) | {"client_hash_id", "content_hash_id", "rf_score",
                           "archetype", "reason_code", "action"}
assert not (queue_cols & {"declined_next_30d", "apr_imp"}), "label leaked into queue"
print("asserted: queue columns are features + score + reasons only — no label, no April window.")


action mix (full pool):
action
no_action            90803
monitor               7567
watch_prepare         1514
investigate_first     1009

top reason codes across the investigate_first tier:
reason_code
ctr_gap+momentum_loss            714
momentum_loss                    285
momentum_loss+volume_exposure      7
base_anchor                        1
volume_exposure                    1

review-first position mix: {'p1': 0.706, 'p2': 0.166, 'top3': 0.101, 'deep': 0.028}



wrote docs/img/w07_action_mix.png
asserted: queue columns are features + score + reasons only — no label, no April window.


## 7. Artifacts the paper embeds

Three charts + one receipt. Everything on the deployed page regenerates from this notebook
(`docs/img/*.png`, relative paths — the page works in any browser and on phones).

In [8]:
# Repository / reproducibility pass: check the exact files the page references.
import os
print("paper = docs/index.html")
ok = True
for f in ["results_precision_at_k.png", "w07_heldout_pk.png", "w07_action_mix.png"]:
    p = DOCIMG / f
    print(f"  img/{f:<30} {os.path.getsize(p):>7} bytes  {'OK' if p.exists() else 'MISSING!'}")
    ok = ok and p.exists()
print("reproducibility chain: docs/ <- work/notebooks/capstone.ipynb <- work/outputs/w0X_*_metrics.json")
print("rerun:  jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=2400 "
      "work/notebooks/capstone.ipynb")
assert ok


paper = docs/index.html
  img/results_precision_at_k.png       76391 bytes  OK
  img/w07_heldout_pk.png               34718 bytes  OK
  img/w07_action_mix.png               29071 bytes  OK
reproducibility chain: docs/ <- work/notebooks/capstone.ipynb <- work/outputs/w0X_*_metrics.json
rerun:  jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=2400 work/notebooks/capstone.ipynb


### Self-check (all boxes honest, and verifyable by running this notebook)

- [x] Section 1–7 filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (executed locally with the month cache)
- [x] Results table recomputed here matches the committed w05 receipt (asserted above)
- [x] The paper's three charts regenerate into `docs/img/` from this notebook
- [x] No client names, URLs, or private queries anywhere — only hashes and aggregates
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] The deployed page has all 9 sections, Abstract on top, Acknowledgments + flyrank.ai link at bottom
- [x] `submission/paper_url.txt` holds the exact deployed URL (one line, nothing else)
- [x] Committed to the repo under `work/notebooks/` — submit the repo URL on the card. Done.

### ML-12 — take this work anywhere in one page

**5-minute demo outline (interview / community / demo day)**
1. 60s — The problem: an editor has ~100 reviews for a portfolio of tens of thousands of pages; "review the
   biggest fallers" scores at chance. Ask the room which page they'd open first.
2. 60s — The data: 100,893 pages, 43 clients, March 2026 facts → April outcome, public anonymized warehouse,
   79M rows of production Search Console facts behind it. Nothing is synthetic.
3. 90s — The method: relative decline label, 9 March-only features, grouped-by-client holdout (never random —
   the random split quietly cheats, 39/39 overlap), leakage audit including the deliberate leak trap.
4. 90s — The result: same split, base 0.43 → P@50 0.86 vs rule 0.44; log odds on momentum; the P@K chart.
5. 60s — The product: four-action queue with human-readable reason codes, limits, monitoring triggers, and a
   no-go list. One honest close: ranking is measured, causal refresh lift is not.


**Social-post cut**
"100,893 pages, 43 clients, real production search data. 9 features from one closed month.
Held-out precision@50 0.86 (base rate 0.43; a 'biggest fallers' rule scores 0.44).
A forest that tells an editor which pages to open first — with the reason on each row.
Values: charts and honest claims in the repo."

**3-sentence employer-facing summary**
I trained a Random Forest on 100,893 pages of March 2026 impressions/clicks/position from a real, public
40-client search warehouse and scored it on 13 fully held-out clients (P@50 0.86, AUC 0.739, base 0.43).
The ranking ships as a human-reviewed action playbook — a short queue with reason codes, limits, no-go rules,
and monitoring triggers — and its numbers, charts, and leakage audits are public on GitHub Pages and in the
repo. What it does not claim: refresh causes recovery; that would need an experiment we didn't run.